In [1]:
from polymer.filter_utils import process_dataset_names, filter_dataset_names
from polymer.datx import convert_datx_dataset
from polymer.tfrecords import create_tfrecord_dataset

In [2]:
config = {
    "input_folder": "/data/sample_pe",
    "output_folder": "/data/processed_pe",
    "tfrecord_folder": "/data/tfrecords_pe",
    "datasets": ["intensity", "quality"],
    "save_dataset_metadata": True,
    "save_instrument_params": True
}

Template: this is what the datxfile names are comprised of, make sure that the separators match exactly.
Filters: if you only want certain data, for example below will only use PE. Anywhere you see 'None' means that it will take all values for that field.

In [3]:
template = "date_polymer-simulant_trial_replicate_day_number_magnification-level"
filters = {
    'date': None,
    'polymer': 'PE',
    'simulant': None,
    'trial': None,
    'replicate': None,
    'day': None,
    'number': None,
    'magnification': None,
    'level': None
}

In [4]:
# First: convert the datx files into something that is more easily processed
convert_datx_dataset(config)

Input: /data/sample_pe
Output: /data/processed_pe
Datasets: ['intensity', 'quality']
Found 8 .datx files


Processing: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 8/8 [00:00<00:00, 68.85it/s]


Dataset index saved to: /data/processed_pe/dataset_index.json
Processed 8 files successfully


In [5]:
# Second: using the results of step one, generate a TFRecord Dataset for training
template = template.replace('-', '_')

samples_to_process = process_dataset_names(config['output_folder'], template)
samples_to_process = filter_dataset_names(samples_to_process, filters)

print(f"Found {len(samples_to_process)} directories\n")

# Uncomment this to see which directories you're converting

# for i, result in enumerate(samples_to_process[:3]):
#     print(f"Directory {i+1}:")
#     print(f"  Name: {result['directory_name']}")
#     print(f"  Path: {result['directory_path']}")
#     for key, value in result.items():
#         if key not in ['directory_path', 'directory_name']:
#             print(f"  {key}: {value}")
#     print()

samples_to_process = [r['directory_name'] for r in samples_to_process]

create_tfrecord_dataset(config['output_folder'], samples_to_process, config['tfrecord_folder'], 5)

Found 8 directories



Creating shards: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████| 2/2 [00:00<00:00,  7.83it/s]


Shard creation complete:
  Shard 1: 5 successful, 0 failed
  Shard 2: 3 successful, 0 failed

Dataset creation complete!
Input samples processed: 8
Total output samples: 8
Failed samples: 0
Created 2 shards in: /data/tfrecords_pe


To check that you have correctly written the TFRecords, run this cell which should list out all the records
Note: change the path to the tfrecord_folder

In [8]:
!ls -ltrh /data/tfrecords_pe

total 65M
-rw-r--r-- 1 root root 25M Oct  2 21:45 record_0001_of_0002.tfrecord
-rw-r--r-- 1 root root 41M Oct  2 21:45 record_0000_of_0002.tfrecord
